# SmileAlign v2, lanjutkan latihan di Kaggle

Notebook ini untuk **melanjutkan** latihan yang terputus di Colab, bukan memulai dari nol.
Colab kena batas pemakaian GPU, Kaggle memberi 30 jam GPU per minggu.

## Sebelum mulai

1. Di Kaggle: **Settings > Accelerator: GPU T4 x2**, **Internet: On**.
2. Di Drive, dua file ini harus diset **Anyone with the link**:
   - `smilealign/runs-smilealign-seg-v2/<run>/weights/last.pt`
   - `smilealign/intraoral_yolov8.zip`
3. Ambil ID dari link share, bagian `drive.google.com/file/d/<ID>/view`.

## Kenapa bisa lanjut tanpa mengulang

`last.pt` menyimpan nomor epoch, state optimizer, dan semua argumen latihan.
Sel 7 memanggil `train(resume=True)` tanpa argumen lain, jadi Ultralytics membaca
semuanya dari checkpoint.

## Tiga hal yang harus persis sama

- **Path dataset.** Checkpoint menyimpan `/content/datasets/mix_v2/data.yaml`.
  Sel 6 membangun dataset di path yang sama, bukan di `/kaggle/working`.
- **Path folder hasil.** Checkpoint menyimpan
  `/content/drive/MyDrive/smilealign/runs-smilealign-seg-v2`.
  Sel 2 membuat symlink ke `/kaggle/working/smilealign` supaya path itu hidup.
- **Pakai `last.pt`, bukan `best.pt`.** `best.pt` tidak menyimpan state optimizer
  dan nomor epoch, resume dari situ mengulang dari epoch 1.

## Sel 0, konfigurasi

In [ ]:
# Isi dua ID file Drive di bawah, ambil dari link share.
# Contoh link: https://drive.google.com/file/d/1AbCdEfGhIjKlMnOpQrStUvWxYz/view
# Maka ID-nya: 1AbCdEfGhIjKlMnOpQrStUvWxYz

FILE_LAST_PT = ""        # last.pt dari folder weights
FILE_INTRAORAL_ZIP = ""  # intraoral_yolov8.zip

# Kosongkan RUN_NAME. Sel 4 membaca nama run langsung dari dalam checkpoint.
RUN_NAME = ""

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/smilealign")
RUNS = ROOT / "runs-smilealign-seg-v2"
STAGING = Path("/kaggle/working/_incoming")

print("ROOT  :", ROOT)
print("RUNS  :", RUNS)
print("siap  :", bool(FILE_LAST_PT and FILE_INTRAORAL_ZIP))

## Sel 1, cek GPU dan pasang paket

In [ ]:
!nvidia-smi

# Ultralytics dipin ke 8.3.* supaya sama dengan versi yang dipakai di Colab.
# Beda versi minor bisa membuat resume menolak checkpoint.
!pip -q install "ultralytics==8.3.*" gdown huggingface_hub

import torch
import ultralytics

print("torch       :", torch.__version__)
print("cuda        :", torch.cuda.is_available())
print("ultralytics :", ultralytics.__version__)

## Sel 2, hidupkan path Drive

Checkpoint menyimpan `project: /content/drive/MyDrive/smilealign/runs-smilealign-seg-v2`.
Path itu tidak ada di Kaggle, jadi kita buat symlink ke folder kerja. Hasilnya
Ultralytics menulis ke folder yang sama persis seperti di Colab, tanpa perlu
mengubah argumen apa pun.

In [ ]:
import os
from pathlib import Path

work = Path("/kaggle/working/smilealign")
work.mkdir(parents=True, exist_ok=True)

Path("/content/drive/MyDrive").mkdir(parents=True, exist_ok=True)
link = Path("/content/drive/MyDrive/smilealign")

if link.is_symlink() or link.exists():
    print("sudah ada:", link, "->", os.path.realpath(link))
else:
    os.symlink(work, link)
    print("dibuat   :", link, "->", os.path.realpath(link))

RUNS.mkdir(parents=True, exist_ok=True)
print("isi RUNS :", [p.name for p in RUNS.iterdir()] or "(kosong)")

## Sel 3, unduh last.pt dan zip intraoral dari Drive

In [ ]:
import gdown

assert FILE_LAST_PT, "FILE_LAST_PT masih kosong, isi dulu di Sel 0"
assert FILE_INTRAORAL_ZIP, "FILE_INTRAORAL_ZIP masih kosong, isi dulu di Sel 0"

STAGING.mkdir(parents=True, exist_ok=True)

# last.pt ditampung dulu karena nama folder run baru diketahui di Sel 4.
staged_last = STAGING / "last.pt"
if not staged_last.exists():
    gdown.download(id=FILE_LAST_PT, output=str(staged_last), quiet=False)
print("last.pt     :", staged_last, round(staged_last.stat().st_size / 1e6, 1), "MB")

zip_path = ROOT / "intraoral_yolov8.zip"
if not zip_path.exists():
    gdown.download(id=FILE_INTRAORAL_ZIP, output=str(zip_path), quiet=False)
print("intraoral   :", zip_path, round(zip_path.stat().st_size / 1e6, 1), "MB")

## Sel 4, verifikasi checkpoint

Gerbang sebelum mengunduh AlphaDent. Kalau di sini `epoch` tidak muncul, atau
`data` tidak menunjuk `/content/datasets/mix_v2`, hentikan dan laporkan.

In [ ]:
import shutil
import torch

ckpt = torch.load(STAGING / "last.pt", map_location="cpu", weights_only=False)
args = ckpt["train_args"]

detected = RUN_NAME or args.get("name") or "train"
RUN_DIR = RUNS / detected
weights_dir = RUN_DIR / "weights"
weights_dir.mkdir(parents=True, exist_ok=True)

last_pt = weights_dir / "last.pt"
if not last_pt.exists():
    shutil.move(str(STAGING / "last.pt"), last_pt)

print("folder run    :", RUN_DIR)
print("epoch terakhir:", ckpt["epoch"])
print("total epochs  :", args["epochs"])
print("data          :", args["data"])
print("imgsz         :", args["imgsz"], "| batch:", args["batch"])
print("project       :", args["project"])
print("dibuat dengan :", ckpt.get("version"))
print()
print("sisa epoch    :", max(args["epochs"] - ckpt["epoch"], 0))

## Sel 5, unduh dan ekstrak AlphaDent

1.455 foto gigi DSLR dari HuggingFace, Apache-2.0. Diunduh ulang karena `/content`
hilang setiap runtime mati.

In [ ]:
import os
import zipfile

from huggingface_hub import hf_hub_download

ALPHA_DIR = "/content/datasets/alphadent"
os.makedirs(ALPHA_DIR, exist_ok=True)

zip_path = hf_hub_download(
    repo_id="ZFTurbo/AlphaDent",
    filename="AlphaDent.zip",
    repo_type="dataset",
    local_dir="/content/ad_zip",
)
print("zip:", zip_path)

with zipfile.ZipFile(zip_path) as z:
    z.extractall(ALPHA_DIR)


def has_struct(root):
    a = os.path.isdir(os.path.join(root, "images", "train")) and os.path.isdir(
        os.path.join(root, "labels", "train")
    )
    b = os.path.isdir(os.path.join(root, "train", "images")) and os.path.isdir(
        os.path.join(root, "train", "labels")
    )
    return bool(a or b)


AD_ROOT = None
for root, dirs, files in os.walk(ALPHA_DIR):
    if has_struct(root):
        AD_ROOT = root
        break
assert AD_ROOT, "Struktur AlphaDent tidak ketemu, cek isi zip"
print("AD_ROOT:", AD_ROOT)

## Sel 6, bangun dataset campuran

Logikanya sama dengan `scripts/prepare_v2_mix.py`, tetapi ditulis di sini supaya
tidak bergantung pada isi repo. Semua kelas dipetakan ke `tooth`, dan baris bbox
AlphaDent diubah menjadi polygon karena Ultralytics menolak file yang mencampur
keduanya.

Tujuannya **wajib** `/content/datasets/mix_v2` karena path itu yang tersimpan di
checkpoint.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

INTRAORAL_DIR = "/content/datasets/intraoral"
MIX_DIR = Path("/content/datasets/mix_v2")

os.makedirs(INTRAORAL_DIR, exist_ok=True)
with zipfile.ZipFile(ROOT / "intraoral_yolov8.zip") as z:
    for info in z.infolist():
        name = info.filename.replace("\\", "/")
        target = os.path.join(INTRAORAL_DIR, name)
        if name.endswith("/"):
            os.makedirs(target, exist_ok=True)
            continue
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with z.open(info) as src, open(target, "wb") as dst:
            dst.write(src.read())
print("intraoral:", os.listdir(os.path.join(INTRAORAL_DIR, "intraoral")))

JOBS = [
    ("io_", Path(INTRAORAL_DIR) / "intraoral"),
    ("ad_", Path(AD_ROOT)),
]
SPLITS = [("train", "train"), ("valid", "valid"), ("test", "valid")]


def pick_dirs(root, split):
    imgs = [p for p in (root / split / "images", root / "images" / split) if p.is_dir()]
    lbls = [p for p in (root / split / "labels", root / "labels" / split) if p.is_dir()]
    return (imgs[0] if imgs else None), (lbls[0] if lbls else None)


def normalize_row(line):
    parts = line.split()
    if len(parts) < 2:
        return None
    coords = parts[1:]
    if len(coords) == 4:
        cx, cy, bw, bh = (float(v) for v in coords)
        x1, y1, x2, y2 = cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2
        corners = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
        return "0 " + " ".join(f"{v:.6f}" for pt in corners for v in pt)
    if len(coords) >= 6 and len(coords) % 2 == 0:
        return "0 " + " ".join(coords)
    return None


counts = {"train": 0, "valid": 0}
dropped = 0
for prefix, root in JOBS:
    for split_src, split_dst in SPLITS:
        imgs, lbls = pick_dirs(root, split_src)
        if imgs is None or lbls is None:
            continue
        (MIX_DIR / split_dst / "images").mkdir(parents=True, exist_ok=True)
        (MIX_DIR / split_dst / "labels").mkdir(parents=True, exist_ok=True)
        for img in sorted(imgs.iterdir()):
            if img.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            lbl = lbls / (img.stem + ".txt")
            if not lbl.exists():
                continue
            rows = []
            for line in lbl.read_text().splitlines():
                t = line.strip()
                if not t:
                    continue
                row = normalize_row(t)
                if row is None:
                    dropped += 1
                    continue
                rows.append(row)
            if not rows:
                continue
            shutil.copy(img, MIX_DIR / split_dst / "images" / f"{prefix}{img.name}")
            (MIX_DIR / split_dst / "labels" / f"{prefix}{img.stem}.txt").write_text(
                "\n".join(rows) + "\n"
            )
            counts[split_dst] += 1

(MIX_DIR / "data.yaml").write_text(
    "train: train/images\nval: valid/images\nnc: 1\nnames: ['tooth']\n"
)
print("merged:", counts, "->", MIX_DIR)
if dropped:
    print("baris label dibuang:", dropped)

## Sel 7, lanjutkan latihan

Tanpa argumen tambahan. `resume=True` membaca ulang semua pengaturan dari
checkpoint, termasuk folder output, jadi latihan lanjut di folder yang sama dan
tidak membuat `train-2`.

Latihan bisa berhenti sendiri sebelum epoch 120 karena `patience=20`. Itu normal,
`best.pt` tetap tersimpan dari epoch terbaik.

In [ ]:
import os

# Kaggle kadang sudah memasang wandb, dan Ultralytics bisa mencoba inisialisasi.
os.environ["WANDB_MODE"] = "disabled"
os.environ["COMET_MODE"] = "disabled"

from ultralytics import YOLO

assert (RUN_DIR / "weights" / "last.pt").exists(), "last.pt tidak ada, ulangi Sel 4"

model = YOLO(str(RUN_DIR / "weights" / "last.pt"))
model.train(resume=True)

## Sel 8, ekspor hasil

In [ ]:
import shutil
from pathlib import Path

out = Path("/kaggle/working/export")
out.mkdir(parents=True, exist_ok=True)

for name in ("best.pt", "last.pt"):
    src = RUN_DIR / "weights" / name
    if src.exists():
        shutil.copy(src, out / name)
        print(name, "->", round(src.stat().st_size / 1e6, 1), "MB")

csv = RUN_DIR / "results.csv"
if csv.exists():
    shutil.copy(csv, out / "results.csv")

print()
print("Folder:", out)
print("Unduh best.pt, lalu taruh di backend/models/seg/best.pt di komputer.")

## Setelah selesai

Unduh `best.pt` dari panel **Output** di kanan, lalu taruh di
`backend/models/seg/best.pt`.

## Kalau terputus lagi

Jalankan ulang Sel 0 sampai Sel 4. Kalau `last.pt` di Drive belum terbarui, ambil
dari Output notebook ini. Sel 5 dan Sel 6 perlu diulang karena `/content` hilang.

## Kalau resume menolak

| Gejala | Sebab | Tindakan |
|---|---|---|
| `FileNotFoundError` data.yaml | Sel 6 belum selesai | ulangi Sel 6, pastikan ada angka `merged:` |
| `RuntimeError` versi Ultralytics | versi beda dari Colab | ulangi Sel 1, pastikan `8.3.x` |
| `KeyError` di train_args | file bukan checkpoint Ultralytics | pastikan yang diunduh `weights/last.pt` |